<a href="https://colab.research.google.com/github/kylermurphy/contigo_edr/blob/main/notebooks/tutorials/CONTIGO_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CONTIGO EDR Example

This notebook goes over a simple example for deriving the energy dissipation rate from the ```contigo``` module.

The first three cells setup the working environment including:
- Installing GMAT the NASA General Mission Analysis Tool
- Retrieving and installing the ```contigo``` module.

**NOTE** that after running the first three cells the session has to be restarted. This is done in the fourth cell. **Because of this you cannot use the ```Run all``` command.

In [ ]:
# Download GMAT (adjust the link for the specific version you need)
!wget -q https://sourceforge.net/projects/gmat/files/GMAT/GMAT-R2025a/gmat-ubuntu-x64-R2025a.tar.gz
!tar -xf gmat-ubuntu-x64-R2025a.tar.gz


In [ ]:
%%bash
cd /content/GMAT/R2025a/api
python BuildApiStartupFile.py

In [ ]:
%%bash
git clone --recursive https://github.com/kylermurphy/contigo_edr.git
cd /content/contigo_edr/
pip install -e .

In [ ]:
exit()

In [ ]:
import logging
import pandas as pd
import numpy as np

# contigo no longer configures logging itself; this shows progress messages
logging.basicConfig(level=logging.INFO)


from contigo.constellation import Constellation
from contigo.edr_efd import EDRDensity


from contigo.ephemeris.spice_ephem import SPICEEphem
from contigo.solar_system_ephem import SolarSystemEnvironment

from contigo.forces.third_body_acc import ThirdBodyEnv
from contigo.forces.grav_pot import EarthPotential
from contigo.forces.srp_gmat import SRPAcc

In [ ]:
sw_e = pd.read_hdf("/content/contigo_edr/data/ESA_pod.hdf") # holds POD info for EDR calculation
sw_o = pd.read_hdf("/content/contigo_edr/data//ore_d.hdf") # holds EDR calculation from other method for comparison


In [ ]:
# create a Constellation object from the ESA POD file
# (the POD positions and velocities are in km and km/s)
hdf_sc = Constellation(state_file=r'/content/contigo_edr/data/ESA_pod.hdf',
                    time_col='DateTime', x_col='x', y_col='y', z_col='z',
                    vx_col='vx', vy_col='vy', vz_col='vz',
                    sc_id_col='filename', sc_fn_slc=slice(-11,-8),
                    tscale_input='GPS', unit_input='km',
                    sc_mass=4.3e+02, cr=1.8, srp_area=1, cd=3.5, drag_area=1.1)

In [ ]:
# setup the ephemeris provider we want
# and the solar system environement
# which defines the tolerance for ephemeris cacheing and the bodies we want
# in our solar system
ephem = SPICEEphem(ephemeris='de440s', frame='ITRF93', observer='EARTH')

env = SolarSystemEnvironment(bodies=['SUN','MOON'], tolerance=0.01, provider=ephem,
                             ephem_time=hdf_sc.sspice_et, gps_time=hdf_sc.sspice_gps,
                             utc_time=hdf_sc.sc_utc)

In [ ]:
# this is a big lmax but this is what
# we use in the orekit derivation and
# is what we need here to for a comparison
ep = EarthPotential(lmax=100)
tba_env = ThirdBodyEnv( )
srp = SRPAcc(apistartup="api_startup_file.txt", gmat_install="/content/GMAT/R2025a")

In [ ]:
edr = EDRDensity(constellation=hdf_sc,
                 solarsys_env= env, force_models=[tba_env,srp],potential_model=ep)

In [ ]:
acc_con = edr.compute_edr()

In [ ]:
den_con = edr.compute_denom()

In [ ]:
contigo = pd.DataFrame(acc_con['ESA'])
# the Orekit EDR is in SI (m^2/s^2), as is the CONTIGO EDR
sw_o['edr_rel'] = sw_o['edr']-sw_o['edr'][0]
ax = contigo['edr'].plot(label='Contigo')
sw_o['edr_rel'].plot(ax=ax,label='Orekit')

ax.legend()

ac = np.allclose(sw_o['edr_rel'].to_numpy(),contigo['edr'].to_numpy(),atol=1E-7,rtol=0.1)
print(ac)

In [ ]:
end = int(5*90*60/10)
ax = contigo['edr'][0:end].plot(label='Contigo')
sw_o['edr_rel'][0:end].plot(ax=ax, label='Orekit')
ax.legend()